In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
# ============================================================
# IMPORTS
# ============================================================

import csv
import random
import numpy as np
import pandas as pd
import networkx as nx
from tqdm import tqdm

from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV
from sklearn.metrics import roc_auc_score, accuracy_score
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier

from xgboost import XGBClassifier


# ============================================================
# 1️⃣ Parse CSV
# ============================================================

def parse_adjacency_list(file_path):
    edges = []

    with open(file_path, 'r') as file:
        reader = csv.reader(file)

        for row in reader:
            if not row:
                continue

            source = str(row[0]).strip()

            for target in row[1:]:
                target = str(target).strip()
                if target:
                    edges.append((source, target))

    return edges


# ============================================================
# 2️⃣ Negative Sampling
# ============================================================

def sample_easy_negatives(G, n):
    nodes = list(G.nodes())
    negatives = set()

    while len(negatives) < n:
        u, v = random.sample(nodes, 2)
        if not G.has_edge(u, v):
            negatives.add((u, v))

    return list(negatives)


# ============================================================
# 3️⃣ Precompute
# ============================================================

def precompute(G):

    in_cache = {}
    out_cache = {}
    degree_cache = {}

    for node in G.nodes():
        u_in = set(G.predecessors(node))
        u_out = set(G.successors(node))

        in_cache[node] = u_in
        out_cache[node] = u_out
        degree_cache[node] = len(u_in | u_out)

    in_degrees = dict(G.in_degree())
    out_degrees = dict(G.out_degree())

    return in_cache, out_cache, degree_cache, in_degrees, out_degrees


# ============================================================
# 4️⃣ FULL 27-FEATURE EXTRACTION
# ============================================================

def extract_features(u, v, G,
                     in_cache, out_cache,
                     degree_cache,
                     in_degrees, out_degrees):

    u_in = in_cache.get(u, set())
    u_out = out_cache.get(u, set())
    v_in = in_cache.get(v, set())
    v_out = out_cache.get(v, set())

    u_all = u_in | u_out
    v_all = v_in | v_out

    u_d = degree_cache.get(u, 0)
    v_d = degree_cache.get(v, 0)

    u_d_in = len(u_in)
    u_d_out = len(u_out)
    u_d_bi = len(u_in & u_out)

    v_d_in = len(v_in)
    v_d_out = len(v_out)
    v_d_bi = len(v_in & v_out)

    common = u_all & v_all
    union = u_all | v_all

    cn_count = len(common)
    jaccard = cn_count / len(union) if union else 0

    transitive_uv = len(u_out & v_in)
    transitive_vu = len(v_out & u_in)
    total_2hop = transitive_uv + transitive_vu

    pref_attach = u_d * v_d

    adamic = 0
    resource = 0

    for z in common:
        deg_z = degree_cache.get(z, 0)
        if deg_z > 1:
            adamic += 1 / np.log(deg_z)
        if deg_z > 0:
            resource += 1 / deg_z

    return [
        u_d, u_d_in, u_d_out, u_d_bi,
        v_d, v_d_in, v_d_out, v_d_bi,
        u_d_in / u_d if u_d else 0,
        u_d_out / u_d if u_d else 0,
        v_d_in / v_d if v_d else 0,
        v_d_out / v_d if v_d else 0,
        cn_count,
        len(u_in & v_in),
        len(u_out & v_out),
        jaccard,
        transitive_uv,
        transitive_vu,
        total_2hop,
        pref_attach,
        abs(u_d - v_d),
        abs(u_d_in - v_d_in),
        abs(u_d_out - v_d_out),
        adamic,
        resource,
        in_degrees.get(v, 0),
        out_degrees.get(u, 0)
    ]


# ============================================================
# 5️⃣ Build Realistic Training Dataset
# ============================================================

def build_training_dataset(train_edges_full):

    positives = random.sample(train_edges_full, 25000)
    remaining_edges = list(set(train_edges_full) - set(positives))

    all_nodes = set()
    for u, v in train_edges_full:
        all_nodes.add(u)
        all_nodes.add(v)

    G = nx.DiGraph()
    G.add_nodes_from(all_nodes)
    G.add_edges_from(remaining_edges)

    negatives = sample_easy_negatives(G, 100000)

    in_cache, out_cache, degree_cache, in_deg, out_deg = precompute(G)

    X = []
    y = []

    print("Building dataset...")

    for u, v in positives:
        X.append(extract_features(u, v, G,
                                  in_cache, out_cache,
                                  degree_cache, in_deg, out_deg))
        y.append(1)

    for u, v in negatives:
        X.append(extract_features(u, v, G,
                                  in_cache, out_cache,
                                  degree_cache, in_deg, out_deg))
        y.append(0)

    return np.array(X), np.array(y)


# ============================================================
# 6️⃣ LOAD DATA
# ============================================================

edges = parse_adjacency_list("/kaggle/input/datasets/yakshithm/project-1/train.csv")
train_edges, _ = train_test_split(edges, test_size=0.2, random_state=42)

X, y = build_training_dataset(train_edges)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

results = {}


# ============================================================
# 7️⃣ Random Forest
# ============================================================

rf = RandomForestClassifier(n_estimators=300,
                            min_samples_leaf=5,
                            n_jobs=-1,
                            random_state=42)

rf.fit(X, y)
rf_probs = rf.predict_proba(X)[:, 1]
rf_preds = rf.predict(X)

results["RandomForest"] = {
    "AUC": roc_auc_score(y, rf_probs),
    "Accuracy": accuracy_score(y, rf_preds)
}


# ============================================================
# 8️⃣ AdaBoost
# ============================================================

ada = AdaBoostClassifier(
    estimator=DecisionTreeClassifier(max_depth=3),
    n_estimators=300,
    random_state=42
)

ada.fit(X, y)
ada_probs = ada.predict_proba(X)[:, 1]
ada_preds = ada.predict(X)

results["AdaBoost"] = {
    "AUC": roc_auc_score(y, ada_probs),
    "Accuracy": accuracy_score(y, ada_preds)
}


# ============================================================
# 9️⃣ XGBoost (GPU)
# ============================================================

xgb = XGBClassifier(
    tree_method="hist",
    device="cuda",
    max_depth=8,
    learning_rate=0.05,
    n_estimators=500,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="logloss",
    random_state=42
)

xgb.fit(X, y)
xgb_probs = xgb.predict_proba(X)[:, 1]
xgb_preds = xgb.predict(X)

results["XGBoost_GPU"] = {
    "AUC": roc_auc_score(y, xgb_probs),
    "Accuracy": accuracy_score(y, xgb_preds)
}


# ============================================================
# 🔟 Print Results
# ============================================================

print("\nModel Results:")
for model_name, metrics in results.items():
    print(f"{model_name}: AUC={metrics['AUC']:.4f}, Accuracy={metrics['Accuracy']:.4f}")


best_model_name = max(results, key=lambda x: results[x]["AUC"])
print("\nBest Model:", best_model_name)

best_model = {"RandomForest": rf,
              "AdaBoost": ada,
              "XGBoost_GPU": xgb}[best_model_name]


# ============================================================
# 1️⃣1️⃣ Build Full Graph for Test Prediction
# ============================================================

all_nodes = set()
for u, v in edges:
    all_nodes.add(u)
    all_nodes.add(v)

G_full = nx.DiGraph()
G_full.add_nodes_from(all_nodes)
G_full.add_edges_from(edges)

in_cache, out_cache, degree_cache, in_deg, out_deg = precompute(G_full)

test_df = pd.read_csv("/kaggle/input/datasets/yakshithm/project-1/test.csv")

X_test = []

for _, row in tqdm(test_df.iterrows(), total=len(test_df)):
    u = str(row["From"]).strip()
    v = str(row["To"]).strip()

    X_test.append(
        extract_features(u, v,
                         G_full,
                         in_cache, out_cache,
                         degree_cache,
                         in_deg, out_deg)
    )

X_test = np.array(X_test)

probs = best_model.predict_proba(X_test)[:, 1]

submission = pd.DataFrame({
    "Id": test_df["Id"],
    "Predictions": probs
})

submission.to_csv("submission.csv", index=False)

print("Submission saved.")